# 4 — Heat

Companion to **section 5**. A heat pump couples *carriers*: it turns the
electricity market and the heat market into one problem, and the exchange
rate between them — the coefficient of performance — moves with the weather.

Runtime: about ten seconds.

In [ ]:
# Make the note's models importable, wherever you launched Jupyter from.
import sys
from pathlib import Path

here = Path.cwd()
note = next(p for p in [here, *here.parents] if (p / "model" / "dispatch.py").exists())
sys.path.insert(0, str(note))
sys.path.insert(0, str(note / "pipeline"))   # the run scripts' helpers
PROCESSED = note / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt

print(f"note root: {note}")

## A coefficient of performance that moves

A heat pump delivers $\mathit{COP}_t$ MWh of heat per MWh of electricity.
The COP falls exactly when it is coldest — which is when heat is most needed.

In [ ]:
from model import dispatch, heat
from run_dispatch_t import dk1_fleet, import_price, PROFILE_OF, YEAR

temp = pd.read_csv(PROCESSED / f"temperature_dk_{YEAR}.csv",
                   index_col="time", parse_dates=True)["temperature_c"]
cop = heat.cop_series(temp)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.scatter(temp, cop, s=2, alpha=0.3, color="#0072B2")
ax.set_xlabel("outdoor temperature (degC)")
ax.set_ylabel("coefficient of performance")
plt.show()

print("COP: min", round(cop.min(), 2), " mean", round(cop.mean(), 2),
      " max", round(cop.max(), 2))

## The two-bus network

Section 3's DK1 instance on the electricity bus, and on the heat bus a
stylised district-heating load built from degree-hours, served by a gas
boiler and an 800 MW (electric) air-sourced heat pump. The horizon is a
contiguous fortnight from 1 January, because the coupling lives in the
heating season.

In [ ]:
from run_heat import HEAT_ANNUAL_TWH, HP_BASE_MW_EL

HOURS = 336

tech = dispatch.read_tech(PROCESSED / "technology_costs_small.csv")
capacity, tech = dk1_fleet(tech)
profiles = pd.read_csv(PROCESSED / f"profiles_dk1_{YEAR}.csv",
                       index_col="time", parse_dates=True)
sample = profiles.iloc[:HOURS]
load = sample["load_mw"].rename("load")
availability = pd.DataFrame({t: sample[c] for t, c in PROFILE_OF.items()})
hourly_cost = import_price(load.index)

temperature = temp.reindex(sample.index, method="nearest")
cop_h = heat.cop_series(temperature)
heat_load = heat.heat_demand(temperature, HEAT_ANNUAL_TWH * len(sample) / len(profiles))

n = heat.build_network(tech, capacity, load, availability, heat_load, cop_h,
                       hp_power_mw_el=HP_BASE_MW_EL, hourly_cost=hourly_cost)
heat.solve(n)

hourly = pd.DataFrame({
    "temperature_c": temperature,
    "cop": cop_h,
    "heat_demand_mw": heat_load,
    "elec_price": n.buses_t.marginal_price["elec"],
    "heat_price": n.buses_t.marginal_price["heat"],
    "hp_heat_mw": -n.links_t.p1["heat_pump"],
    "boiler_mw": n.generators_t.p["gas_boiler"],
})
hourly.describe().round(1)

## Two prices, one exchange rate

The heat price is a dual like any other. When the heat pump is the marginal
source of heat, $\lambda^{\mathrm{heat}}_t = \lambda^{\mathrm{elec}}_t /
\mathit{COP}_t$ — the electricity price converted at the hour's exchange
rate. When the pump is at capacity and the boiler is marginal, the heat price
is the boiler's fuel cost and stops tracking electricity altogether.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(hourly.index, hourly["elec_price"], color="#0072B2", lw=1.1, label="electricity")
axes[0].plot(hourly.index, hourly["heat_price"], color="#D55E00", lw=1.1, label="heat")
axes[0].set_ylabel("price (EUR/MWh)")
axes[0].legend(frameon=False)
axes[1].stackplot(hourly.index, hourly["hp_heat_mw"], hourly["boiler_mw"],
                  labels=["heat pump", "gas boiler"], colors=["#009E73", "#999999"], alpha=0.8)
axes[1].set_ylabel("heat supplied (MW)")
axes[1].legend(frameon=False, loc="upper left")
fig.autofmt_xdate()
plt.show()

pump_marginal = hourly["boiler_mw"] < 1.0
implied = hourly["elec_price"] / hourly["cop"]
print("hours with the pump marginal:", int(pump_marginal.sum()), "of", len(hourly))
print("mean |heat price - elec price / COP| in those hours:",
      round((hourly["heat_price"] - implied)[pump_marginal].abs().mean(), 2), "EUR/MWh")

## Rolling out heat pumps

Sweep the pump's capacity. More pumps means more electricity demand in cold,
low-COP hours — the correlation is the whole point — so watch what happens to
the *electricity* price as the heat sector electrifies.

In [ ]:
rows = []
for mw in [0.0, 400.0, 800.0, 1600.0]:
    m = heat.build_network(tech, capacity, load, availability, heat_load, cop_h,
                           hp_power_mw_el=mw, hourly_cost=hourly_cost)
    heat.solve(m)
    hp_heat = -m.links_t.p1["heat_pump"].sum() if mw > 0 else 0.0
    rows.append({
        "heat pump MW_el": mw,
        "share of heat from pump": hp_heat / heat_load.sum(),
        "mean elec price": m.buses_t.marginal_price["elec"].mean(),
        "mean heat price": m.buses_t.marginal_price["heat"].mean(),
    })
pd.DataFrame(rows).set_index("heat pump MW_el").round(2)

## Which way does it bias the answer?

Suppose you had used a single seasonal COP instead of the hourly curve. Work
out the direction of the error **before** running anything — then check.

The heat pump runs hardest when it is coldest; cold hours are low-COP hours;
so the hours that carry the most weight are the hours where the true COP is
below the annual mean. A constant COP therefore *overstates* the pump's
efficiency where it matters, and so overstates its value.

## Your turn

1. Compute the heat-demand-weighted average COP and compare it with the
   simple average. Then re-solve with `cop_h` replaced by that constant and
   compare the pump's share of heat with the table above.
2. Add a carbon price on both sides of the coupling:
   `heat.build_network(..., co2_price=85)`. Which fuel does it hit harder,
   the boiler's gas or the power fleet's?
3. Move the fortnight to July (`profiles.iloc[4344:4344 + HOURS]`, and the
   matching temperature). Does the heat price still track electricity?

In [ ]:
# Try it here.